# 🚗 Detección de Coches con YOLOv8

Este notebook entrena un modelo de detección de objetos para identificar coches usando YOLOv8 de Ultralytics.

## Dataset
- **Imágenes**: ~9000 imágenes de coches
- **Anotaciones**: Archivos .txt en formato YOLO (class_id x_center y_center width height)
- **Clases**: 1 clase ("car")
- **Estructura**: `vehicle-detection-3/train` y `vehicle-detection-3/val`

## 1️⃣ Instalación de Dependencias

Instalamos Ultralytics (que incluye YOLOv8) y otras librerías necesarias.

In [ ]:
# Instalar dependencias mínimas (sin Ultralytics)
# Nota: usa tensorflow-cpu para menor peso si no hay GPU
%pip install -q tensorflow-cpu scikit-learn pandas matplotlib seaborn pyyaml pillow opencv-python

import sys
print("✅ Entorno listo. Versión de Python:", sys.version)
try:
    import tensorflow as tf
    print("TensorFlow:", tf.__version__)
except Exception as e:
    print("⚠️ TensorFlow no disponible:", e)

## 2️⃣ Importar Librerías

Importamos las librerías necesarias para el entrenamiento y visualización.

In [ ]:
from ultralytics import YOLO
import os
import yaml
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from PIL import Image

print("✅ Librerías importadas correctamente")

## 3️⃣ Configuración de Rutas del Dataset

Definimos las rutas del dataset. La estructura esperada es:
```
vehicle-detection-3/
├── train/
│   ├── images/
│   └── labels/
└── val/
    ├── images/
    └── labels/
```

In [ ]:
# Configuración robusta de rutas del dataset (similar a crai.ipynb pero simplificado)
from pathlib import Path
import os

# Ruta al dataset (ajusta si es necesario)
DATASET_PATH = Path("../../vehicle-detection-3").resolve()
print(f"📁 DATASET_PATH: {DATASET_PATH}")

# Detectar estructura de carpetas
roboflow_layout = (
    (DATASET_PATH / "train" / "images").is_dir() and
    (DATASET_PATH / "train" / "labels").is_dir() and
    (
        (DATASET_PATH / "valid" / "images").is_dir() and (DATASET_PATH / "valid" / "labels").is_dir()
        or (DATASET_PATH / "val" / "images").is_dir() and (DATASET_PATH / "val" / "labels").is_dir()
    )
)
ua_detrac_style_layout = (
    (DATASET_PATH / "images" / "train").is_dir() and
    (DATASET_PATH / "labels" / "train").is_dir()
)

# Asignar rutas según layout
if roboflow_layout:
    train_images = DATASET_PATH / "train" / "images"
    train_labels = DATASET_PATH / "train" / "labels"
    # Preferir 'valid' si existe; si no, usar 'val'
    if (DATASET_PATH / "valid" / "images").is_dir():
        val_images = DATASET_PATH / "valid" / "images"
        val_labels = DATASET_PATH / "valid" / "labels"
        val_split_name = "valid"
    else:
        val_images = DATASET_PATH / "val" / "images"
        val_labels = DATASET_PATH / "val" / "labels"
        val_split_name = "val"
elif ua_detrac_style_layout:
    train_images = DATASET_PATH / "images" / "train"
    train_labels = DATASET_PATH / "labels" / "train"
    # Validación puede ser 'val' o inexistente; si no existe, usamos train como fallback
    if (DATASET_PATH / "images" / "val").is_dir() and (DATASET_PATH / "labels" / "val").is_dir():
        val_images = DATASET_PATH / "images" / "val"
        val_labels = DATASET_PATH / "labels" / "val"
        val_split_name = "val"
    else:
        print("⚠️ Split de validación no encontrado; usando train como validación.")
        val_images = train_images
        val_labels = train_labels
        val_split_name = "train"
else:
    raise FileNotFoundError(f"❌ Estructura de dataset no reconocida en: {DATASET_PATH}")

# Verificación estricta de existencia
required_paths = [DATASET_PATH, train_images, train_labels, val_images, val_labels]
if not all(p.exists() for p in required_paths):
    missing = [str(p) for p in required_paths if not p.exists()]
    raise FileNotFoundError(
        "❌ ERROR: Faltan rutas del dataset:\n" + "\n".join(f" - {m}" for m in missing)
    )

# Resumen
print("\n✅ Rutas verificadas:")
print(f" - Train images: {train_images}")
print(f" - Train labels: {train_labels}")
print(f" - {val_split_name.capitalize()} images: {val_images}")
print(f" - {val_split_name.capitalize()} labels: {val_labels}")

# Conteo rápido
def count_files(path: Path, pattern: str) -> int:
    return len(list(path.glob(pattern)))

print("\n📊 Conteo:")
print(f"  - Train images: {count_files(train_images, '*.jpg')}")
print(f"  - Train labels: {count_files(train_labels, '*.txt')}")
print(f"  - {val_split_name.capitalize()} images: {count_files(val_images, '*.jpg')}")
print(f"  - {val_split_name.capitalize()} labels: {count_files(val_labels, '*.txt')}")

## 4️⃣ Crear Archivo data.yaml

YOLOv8 necesita un archivo de configuración que especifique las rutas y clases del dataset.

In [ ]:
# Crear el archivo data.yaml adaptado al layout detectado
from pathlib import Path
import yaml

# Construir rutas relativas a DATASET_PATH
train_rel = train_images.relative_to(DATASET_PATH)
val_rel = val_images.relative_to(DATASET_PATH)

data_yaml_content = {
    'path': str(DATASET_PATH),
    'train': str(train_rel).replace('\\', '/'),
    'val': str(val_rel).replace('\\', '/'),
    'names': {
        0: 'car'
    }
}

yaml_path = DATASET_PATH / "data.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, sort_keys=False)

print(f"✅ Archivo data.yaml creado en: {yaml_path}")
print("\nContenido:")
print(yaml.dump(data_yaml_content, sort_keys=False))

## 5️⃣ Visualizar Ejemplos del Dataset

Antes de entrenar, visualicemos algunas imágenes con sus anotaciones para verificar que todo está correcto.

In [ ]:
def visualize_sample(image_path, label_path):
    """
    Visualiza una imagen con sus bounding boxes.
    """
    # Leer imagen
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    # Leer anotaciones (formato YOLO: class_id x_center y_center width height)
    with open(label_path, 'r') as f:
        lines = f.readlines()
    
    # Dibujar bounding boxes
    for line in lines:
        class_id, x_center, y_center, width, height = map(float, line.strip().split())
        
        # Convertir coordenadas normalizadas a píxeles
        x_center *= w
        y_center *= h
        width *= w
        height *= h
        
        # Calcular esquinas del bounding box
        x1 = int(x_center - width/2)
        y1 = int(y_center - height/2)
        x2 = int(x_center + width/2)
        y2 = int(y_center + height/2)
        
        # Dibujar rectángulo
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, 'car', (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    return img

# Visualizar 3 ejemplos aleatorios
train_images_list = list(train_images.glob('*.jpg'))[:3]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for idx, img_path in enumerate(train_images_list):
    label_path = train_labels / f"{img_path.stem}.txt"
    if label_path.exists():
        img_with_boxes = visualize_sample(img_path, label_path)
        axes[idx].imshow(img_with_boxes)
        axes[idx].set_title(f"Ejemplo {idx+1}")
        axes[idx].axis('off')

plt.tight_layout()
plt.show()
print("✅ Visualización completada")

## 6️⃣ Cargar Modelo Pre-entrenado YOLOv8

Utilizamos YOLOv8n (nano), que es el modelo más pequeño y rápido. Otras opciones:
- `yolov8s.pt` - Small (más preciso, más lento)
- `yolov8m.pt` - Medium
- `yolov8l.pt` - Large
- `yolov8x.pt` - Extra Large

In [ ]:
# Cargar modelo pre-entrenado YOLOv8 nano
# Se descargará automáticamente si no existe localmente
model = YOLO('yolov8n.pt')

print("✅ Modelo YOLOv8n cargado correctamente")
print(f"Modelo: {model.model}")

## 7️⃣ Entrenar el Modelo

Entrenamos el modelo con parámetros simples:
- **epochs**: 50 épocas (puedes ajustar según tiempo disponible)
- **imgsz**: 640 (tamaño de imagen estándar)
- **batch**: -1 (auto-batch, se ajusta automáticamente según GPU/RAM)
- **patience**: 10 (early stopping si no mejora en 10 épocas)
- **device**: 0 para GPU, 'cpu' para CPU

In [ ]:
# Entrenar el modelo
results = model.train(
    data=str(yaml_path),     # Ruta al archivo data.yaml
    epochs=50,               # Número de épocas
    imgsz=640,              # Tamaño de imagen
    batch=-1,               # Auto-batch
    patience=10,            # Early stopping
    device=0,               # GPU (cambiar a 'cpu' si no tienes GPU)
    project='runs/detect',  # Carpeta donde se guardarán los resultados
    name='car_detection',   # Nombre del experimento
    exist_ok=True,          # Sobrescribir si existe
    verbose=True            # Mostrar progreso detallado
)

print("\n✅ Entrenamiento completado!")
print(f"Los resultados se guardaron en: runs/detect/car_detection")

## 8️⃣ Visualizar Resultados del Entrenamiento

YOLOv8 genera automáticamente gráficos de las métricas de entrenamiento.

In [ ]:
# Visualizar gráficas de entrenamiento
results_path = Path("runs/detect/car_detection")

# Mostrar la curva de resultados
results_img = results_path / "results.png"
if results_img.exists():
    img = Image.open(results_img)
    plt.figure(figsize=(16, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title("Métricas de Entrenamiento")
    plt.show()
else:
    print("⚠️ Archivo results.png no encontrado")

print("\n📊 Métricas disponibles en:", results_path)

## 9️⃣ Validación del Modelo

Evaluamos el modelo en el conjunto de validación para obtener métricas de rendimiento.

In [ ]:
# Cargar el mejor modelo entrenado
best_model_path = results_path / "weights" / "best.pt"
model_best = YOLO(best_model_path)

# Validar el modelo
metrics = model_best.val()

# Mostrar métricas principales
print("\n📊 Métricas de Validación:")
print(f"  - mAP50: {metrics.box.map50:.4f}")         # mAP @ IoU=0.50
print(f"  - mAP50-95: {metrics.box.map:.4f}")        # mAP @ IoU=0.50:0.95
print(f"  - Precision: {metrics.box.mp:.4f}")        # Precisión media
print(f"  - Recall: {metrics.box.mr:.4f}")           # Recall medio

print("\n✅ Validación completada")

## 🔟 Inferencia en Nuevas Imágenes

Probamos el modelo entrenado en imágenes del conjunto de validación.

In [ ]:
# Seleccionar algunas imágenes de validación
test_images = list(val_images.glob('*.jpg'))[:6]

# Hacer predicciones
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, img_path in enumerate(test_images):
    # Hacer predicción
    results = model_best.predict(
        source=str(img_path),
        conf=0.25,          # Umbral de confianza
        iou=0.45,           # Umbral de IoU para NMS
        verbose=False
    )
    
    # Obtener imagen con predicciones dibujadas
    img_with_predictions = results[0].plot()
    
    # Convertir de BGR a RGB
    img_rgb = cv2.cvtColor(img_with_predictions, cv2.COLOR_BGR2RGB)
    
    # Mostrar
    axes[idx].imshow(img_rgb)
    axes[idx].set_title(f"Predicción {idx+1}")
    axes[idx].axis('off')

plt.tight_layout()
plt.show()
print("✅ Inferencia completada")

## 1️⃣1️⃣ Guardar y Exportar el Modelo

Guardamos el modelo entrenado y lo exportamos a diferentes formatos si es necesario.

In [ ]:
# El modelo ya está guardado en runs/detect/car_detection/weights/
print(f"📦 Modelos guardados:")
print(f"  - Mejor modelo: {best_model_path}")
print(f"  - Último modelo: {results_path / 'weights' / 'last.pt'}")

# Opcional: Exportar a ONNX para despliegue en producción
# model_best.export(format='onnx')
# print("✅ Modelo exportado a ONNX")

# Para usar el modelo en el futuro:
print("\n💡 Para cargar el modelo entrenado:")
print(f"   from ultralytics import YOLO")
print(f"   model = YOLO('{best_model_path}')")
print(f"   results = model.predict('imagen.jpg')")

print("\n✅ Notebook completado exitosamente!")

---

## 📝 Resumen

Este notebook ha cubierto:

1. ✅ **Instalación** de Ultralytics YOLOv8
2. ✅ **Configuración** del dataset en formato YOLO
3. ✅ **Visualización** de ejemplos con anotaciones
4. ✅ **Entrenamiento** del modelo con YOLOv8n
5. ✅ **Evaluación** con métricas (mAP, Precision, Recall)
6. ✅ **Inferencia** en nuevas imágenes
7. ✅ **Exportación** del modelo entrenado

### 🎯 Próximos pasos

- **Mejorar el modelo**: Prueba con YOLOv8s o YOLOv8m para mayor precisión
- **Ajustar hiperparámetros**: Experimenta con learning rate, batch size, epochs
- **Data augmentation**: YOLOv8 ya incluye augmentation por defecto, pero puedes personalizarlo
- **Desplegar el modelo**: Exporta a ONNX, TensorRT o CoreML para producción
- **Video en tiempo real**: Usa el modelo para detección en streams de video

### 📚 Recursos

- [Documentación Ultralytics](https://docs.ultralytics.com/)
- [YOLOv8 GitHub](https://github.com/ultralytics/ultralytics)

# 🧠 Agente Multimodal: Imágenes, Series Temporales y Texto
Este bloque implementa un agente con tres flujos separados (imágenes, series temporales y texto), con preprocesado, modelado y validación avanzados, evitando Ultralytics. Se usan técnicas como early stopping, regularización, validación cruzada y split temporal.

In [ ]:
# ️⃣ Imágenes: Clasificación binaria basada en etiquetas YOLO (car presente)
import os
from pathlib import Path
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import cv2
from tqdm import tqdm
import matplotlib.pyplot as plt

# Semillas para reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# A partir de train_images/train_labels detectados antes, creamos etiquetas a nivel imagen:
def build_image_class_list(images_dir: Path, labels_dir: Path, exts=(".jpg", ".jpeg", ".png")) -> list[tuple[Path, int]]:
    data = []
    for img_path in images_dir.iterdir():
        if img_path.suffix.lower() not in exts:
            continue
        label_path = labels_dir / f"{img_path.stem}.txt"
        y = 0
        if label_path.exists():
            with open(label_path, "r") as f:
                # si hay al menos una anotación, consideramos 'car presente'
                y = 1 if len([ln for ln in f.readlines() if ln.strip()]) > 0 else 0
        data.append((img_path, y))
    return data

dataset_list = build_image_class_list(train_images, train_labels)
random.shuffle(dataset_list)
print("Total imágenes (train):", len(dataset_list), "| Positivas:", sum(y for _, y in dataset_list))

# Split train/val
val_ratio = 0.2
val_size = int(len(dataset_list) * val_ratio)
val_list = dataset_list[:val_size]
train_list = dataset_list[val_size:]
print(f"Split -> Train: {len(train_list)}, Val: {len(val_list)}")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def decode_and_resize(img_path: tf.Tensor) -> tf.Tensor:
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img

def augment(img: tf.Tensor) -> tf.Tensor:
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.1)
    img = tf.image.random_contrast(img, 0.9, 1.1)
    return img

def make_ds(pairs: list[tuple[Path, int]], training: bool) -> tf.data.Dataset:
    xs = [str(p) for p, _ in pairs]
    ys = [y for _, y in pairs]
    ds_x = tf.data.Dataset.from_tensor_slices(xs)
    ds_y = tf.data.Dataset.from_tensor_slices(ys)
    ds = tf.data.Dataset.zip((ds_x, ds_y))
    def _map(x, y):
        img = decode_and_resize(x)
        if training:
            img = augment(img)
        return img, tf.cast(y, tf.float32)
    ds = ds.shuffle(1024, seed=SEED) if training else ds
    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(train_list, training=True)
val_ds = make_ds(val_list, training=False)

# Modelo CNN pequeño con regularización y early stopping
def build_cnn(input_shape=(224,224,3)) -> keras.Model:
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, padding='same', activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4))(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, padding='same', activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['AUC','Precision','Recall'])
    return model

cnn = build_cnn()
early = keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_auc', mode='max')
history = cnn.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=[early], verbose=1)

# Evaluación final
eval_metrics = cnn.evaluate(val_ds, verbose=0)
print("✅ Eval imagenes (loss, AUC, Precision, Recall):", eval_metrics)

# Guardar modelo
MODELS_DIR = Path("../../ai_models").resolve()
(MODELS_DIR / "image").mkdir(parents=True, exist_ok=True)
IMG_MODEL_PATH = MODELS_DIR / "image" / "cnn_car_presence.keras"
cnn.save(IMG_MODEL_PATH)
print("💾 Modelo de imágenes guardado en:", IMG_MODEL_PATH)

In [ ]:
# ️⃣ Series Temporales: LSTM con validación temporal
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
from pathlib import Path

# Intentar cargar dataset de series temporales; si no existe, generar sintético
TS_DIR = Path("../../data/timeseries")
TS_DIR.mkdir(parents=True, exist_ok=True)
ts_files = list(TS_DIR.glob("*.csv"))

if len(ts_files) == 0:
    # Serie sintética (seno + ruido)
    T = 1500
    t = np.arange(T)
    series = np.sin(0.02 * t) + 0.5*np.sin(0.05*t + 1.2) + 0.1*np.random.randn(T)
    df_ts = pd.DataFrame({"value": series})
    df_ts.to_csv(TS_DIR / "synthetic_series.csv", index=False)
    ts_files = [TS_DIR / "synthetic_series.csv"]
    print("⚠️ No había CSVs. Generado: synthetic_series.csv")
else:
    print("📁 Archivos de series:", [p.name for p in ts_files])

df = pd.read_csv(ts_files[0])
assert 'value' in df.columns, "El CSV debe tener una columna 'value'"
values = df['value'].values.astype('float32')
values = values.reshape(-1, 1)

# Normalización
scaler = StandardScaler()
values_scaled = scaler.fit_transform(values)

# Ventaneo
WINDOW = 32
HORIZON = 1
def make_xy(arr, window=WINDOW, horizon=HORIZON):
    X, y = [], []
    for i in range(len(arr) - window - horizon + 1):
        X.append(arr[i:i+window, 0])
        y.append(arr[i+window:i+window+horizon, 0])
    return np.array(X)[..., None], np.array(y)
X, y = make_xy(values_scaled)
print("X shape:", X.shape, "y shape:", y.shape)

# Split temporal: 80/20
split = int(0.8 * len(X))
X_train, y_train = X[:split], y[:split]
X_val, y_val = X[split:], y[split:]

# Modelo LSTM compacto con regularización
def build_lstm(input_shape: tuple) -> keras.Model:
    inputs = keras.Input(shape=input_shape)
    x = layers.LSTM(64, return_sequences=False, dropout=0.2, recurrent_dropout=0.0)(inputs)
    x = layers.Dense(32, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    outputs = layers.Dense(HORIZON)(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return model

lstm = build_lstm((WINDOW, 1))
early_ts = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_mae', mode='min')
hist_ts = lstm.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100, batch_size=64, callbacks=[early_ts], verbose=0)
eval_ts = lstm.evaluate(X_val, y_val, verbose=0)
print("✅ Eval time series (loss, MAE):", eval_ts)

# Guardar modelo y scaler
(MODELS_DIR / "timeseries").mkdir(parents=True, exist_ok=True)
TS_MODEL_PATH = MODELS_DIR / "timeseries" / "lstm_forecaster.keras"
SCALER_PATH = MODELS_DIR / "timeseries" / "scaler.npy"
lstm.save(TS_MODEL_PATH)
np.save(SCALER_PATH, {"mean_": scaler.mean_, "scale_": scaler.scale_})
print("💾 Modelo de series guardado en:", TS_MODEL_PATH)

In [ ]:
# ️⃣ Texto: Clasificación con TF-IDF + Regresión Logística y K-Fold
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
import numpy as np
from pathlib import Path
import joblib

TXT_DIR = Path("../../data/text")
TXT_DIR.mkdir(parents=True, exist_ok=True)
txt_files = list(TXT_DIR.glob("*.csv"))

if len(txt_files) == 0:
    df_text = pd.DataFrame({
        "text": [
            "El coche es rápido y eficiente.",
            "Este camión es muy lento.",
            "Me encanta conducir mi coche nuevo.",
            "El transporte público es barato.",
            "Los coches eléctricos son el futuro.",
            "Odio los atascos en la ciudad.",
        ],
        "label": [1, 0, 1, 0, 1, 0],
    })
    df_text.to_csv(TXT_DIR / "synthetic_text.csv", index=False)
    txt_files = [TXT_DIR / "synthetic_text.csv"]
    print("⚠️ No había CSVs. Generado: synthetic_text.csv")
else:
    print("📁 Archivos de texto:", [p.name for p in txt_files])

df_txt = pd.read_csv(txt_files[0])
assert {'text','label'}.issubset(set(df_txt.columns)), "El CSV debe tener columnas 'text' y 'label'"

X = df_txt['text'].astype(str).values
y = df_txt['label'].astype(int).values

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
clf = LogisticRegression(max_iter=200)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
f1s = []
for fold, (tr, va) in enumerate(skf.split(X, y), 1):
    Xtr = vectorizer.fit_transform(X[tr])
    Xva = vectorizer.transform(X[va])
    clf.fit(Xtr, y[tr])
    pred = clf.predict(Xva)
    f1 = f1_score(y[va], pred)
    f1s.append(f1)
    print(f"Fold {fold} F1: {f1:.4f}")
print(f"✅ F1 media (5 folds): {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")

# Entrenar en todo el dataset y guardar
Xall = vectorizer.fit_transform(X)
clf.fit(Xall, y)
(MODELS_DIR / "text").mkdir(parents=True, exist_ok=True)
VEC_PATH = MODELS_DIR / "text" / "tfidf.joblib"
TXT_MODEL_PATH = MODELS_DIR / "text" / "logreg.joblib"
joblib.dump(vectorizer, VEC_PATH)
joblib.dump(clf, TXT_MODEL_PATH)
print("💾 Modelo de texto guardado en:", TXT_MODEL_PATH)